# Day 064 — Exercise 3: Template Engine

A template engine turns parameterised strings into full prompts. A product ships with a curated set of templates (email, tweet, blog intro, summary) and lets users fill in the blanks — much lower friction than writing a prompt from scratch.

```
TEMPLATES["email"] = "Write a {tone} email to {recipient} about {topic}."

render_template(TEMPLATES["email"], tone="professional",
                recipient="Alice", topic="AI")
# → 'Write a professional email to Alice about AI.'
```

In [ ]:
import re

TEMPLATES = {
    "email":      "Write a {tone} email to {recipient} about {topic}.",
    "tweet":      "Write a {tone} tweet about {topic} in under 280 characters.",
    "summary":    "Write a concise {length}-sentence summary of: {content}",
    "blog_intro": "Write a blog intro about {topic} for a {audience} audience.",
}


## Task

Implement `render_template(template_str, **vars) -> str`:

1. Find all `{key}` placeholders with `re.findall(r'\{(\w+)\}', ...)`
2. Compute `missing = required_keys - set(vars.keys())`
3. If missing: `raise ValueError(f'Missing template variables: {missing}')`
4. Replace each `{key}` with `str(vars[key])`
5. Return the rendered string (extra kwargs silently ignored)

## Your Implementation

In [ ]:
def render_template(template_str: str, **vars) -> str:
    """Replace {key} placeholders in template_str.

    - All {key} placeholders in template_str must have matching kwargs.
    - Missing variable -> raise ValueError('Missing template variables: {...}')
    - Extra kwargs are silently ignored.
    - Use re.findall(r'\\{(\\w+)\\}', template_str) to find required keys.
    """
    # TODO: find required keys, check for missing, replace placeholders
    raise NotImplementedError


In [ ]:
def render_template(template_str: str, **vars) -> str:
    required = set(re.findall(r'\{(\w+)\}', template_str))
    missing  = required - set(vars.keys())
    if missing:
        raise ValueError(f"Missing template variables: {missing}")
    result = template_str
    for key, value in vars.items():
        result = result.replace(f"{{{key}}}", str(value))
    return result


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # basic substitution
    t = "Write a {tone} email to {recipient} about {topic}."
    r = render_template(t, tone="professional", recipient="Alice", topic="AI")
    assert r == "Write a professional email to Alice about AI."
    score += 1; print("\u2705 basic substitution works")

    # all placeholders in TEMPLATES are replaceable
    tweet = render_template(TEMPLATES["tweet"], tone="casual", topic="Python")
    assert "casual" in tweet and "Python" in tweet
    score += 1; print("\u2705 renders a TEMPLATES entry correctly")

    # missing variable → ValueError
    try:
        render_template(t, tone="casual")  # missing recipient, topic
        assert False, "should raise ValueError"
    except ValueError as e:
        assert "Missing" in str(e)
    score += 1; print("\u2705 missing variable \u2192 ValueError")

    # extra kwargs are ignored
    r2 = render_template("Hello {name}", name="World", extra="ignored")
    assert r2 == "Hello World"
    score += 1; print("\u2705 extra kwargs are silently ignored")

    # numeric value is stringified
    r3 = render_template(TEMPLATES["summary"], length=3, content="some text")
    assert "3" in r3
    score += 1; print("\u2705 numeric value is converted to string")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def render_template(template_str: str, **vars) -> str:
    required = set(re.findall(r'\{(\w+)\}', template_str))
    missing  = required - set(vars.keys())
    if missing:
        raise ValueError(f"Missing template variables: {missing}")
    result = template_str
    for key, value in vars.items():
        result = result.replace(f"{{{key}}}", str(value))
    return result
```

**Why not Python's `str.format_map`?** `'hello {name}'.format_map(vars)` raises `KeyError` on missing keys but doesn't tell you *which* keys are missing. Our version collects all missing keys at once and reports them all in one error — much friendlier for a UI that shows the user what fields to fill in.

</details>